In [ ]:
!pip install python-chess torch numpy
!apt-get update
!apt-get install -y stockfish

import os
import torch
import torch.nn as nn
import torch.optim as optim
import chess
import chess.engine
import numpy as np
from IPython.display import display, SVG, clear_output
import ipywidgets as widgets

# Configuration
STOCKFISH_PATH = "/usr/games/stockfish"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class ChessPolicyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(13, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Flatten()
        )
        self.fc = nn.Linear(64 * 8 * 8, 4096)

    def forward(self, x):
        return self.fc(self.conv(x))

def board_to_tensor(board):
    tensor = np.zeros((13, 8, 8), dtype=np.float32)
    for square, piece in board.piece_map().items():
        row, col = divmod(square, 8)
        plane = piece.piece_type - 1 + (6 if piece.color == chess.BLACK else 0)
        tensor[plane, row, col] = 1.0
    tensor[12, :, :] = 1.0 if board.turn == chess.WHITE else 0.0
    return torch.from_numpy(tensor).unsqueeze(0).to(DEVICE)

def get_legal_mask(board):
    mask = torch.zeros(4096).to(DEVICE)
    for move in board.legal_moves:
        idx = move.from_square * 64 + move.to_square
        mask[idx] = 1.0
    return mask

def sample_move(logits, mask, temperature=1.0, top_p=0.9):
    """
    Applies temperature scaling and Top-p (Nucleus) filtering to the logits.
    """
    # 1. Apply Mask and Temperature
    logits = (logits / max(temperature, 1e-6))
    logits = logits + (1.0 - mask) * -1e9 # Mask illegal moves

    probs = torch.softmax(logits, dim=-1)

    # 2. Top-p (Nucleus) Filtering
    sorted_probs, sorted_indices = torch.sort(probs, descending=True, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)

    # Remove tokens with cumulative probability above the threshold
    sorted_indices_to_remove = cumulative_probs > top_p
    # Shift the indices to keep the first token that exceeds top_p
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = 0

    indices_to_remove = sorted_indices_to_remove.scatter(1, sorted_indices, sorted_indices_to_remove)
    probs[indices_to_remove] = 0.0

    # 3. Re-normalize and Sample
    probs = probs / probs.sum(dim=-1, keepdim=True)
    move_idx = torch.multinomial(probs, num_samples=1).item()
    return move_idx

def supervised_pretrain(model, optimizer, engine, iterations=1000):
    model.train()
    criterion = nn.CrossEntropyLoss()
    print("--- Starting Supervised Pre-training ---")

    i = 0
    while i < iterations:
        board = chess.Board()
        # Randomize start
        for _ in range(np.random.randint(0, 25)):
            if board.is_game_over(): break
            board.push(np.random.choice(list(board.legal_moves)))

        # If the randomized board is finished, skip this iteration
        if board.is_game_over():
            continue

        result = engine.analyse(board, chess.engine.Limit(depth=10))

        # SAFETY CHECK: Ensure 'pv' exists and is not empty
        if "pv" in result and len(result["pv"]) > 0:
            best_move = result["pv"][0]
            target_idx = torch.tensor([best_move.from_square * 64 + best_move.to_square]).to(DEVICE)

            logits = model(board_to_tensor(board))
            loss = criterion(logits, target_idx)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            if i % 100 == 0:
                print(f"SL Iter {i:04d} | Loss: {loss.item():.4f}")
            i += 1 # Only increment if we successfully processed a move
        else:
            # If no move was found, just try a new board state
            continue

def train_grpo_enhanced(model, optimizer, engine, group_size=32, iterations=1000):
    model.train()
    board = chess.Board()
    print("\n--- Starting GRPO Reinforcement Learning ---")
    for i in range(iterations):
        state_tensor = board_to_tensor(board)
        logits = model(state_tensor)
        mask = get_legal_mask(board)

        # Sampling for GRPO
        probs = torch.softmax(logits + (1.0 - mask) * -1e9, dim=-1)
        sampled_indices = torch.multinomial(probs, num_samples=group_size, replacement=True).squeeze()

        rewards = []
        log_probs = []
        for idx in sampled_indices:
            move = chess.Move(idx.item() // 64, idx.item() % 64)
            log_probs.append(torch.log(torch.softmax(logits, dim=-1)[0, idx] + 1e-8))
            info = engine.analyse(board, chess.engine.Limit(depth=8), root_moves=[move])
            rewards.append(info["score"].relative.score(mate_score=10000) / 100.0)

        rewards = torch.tensor(rewards, dtype=torch.float32).to(DEVICE)
        advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-8)
        loss = -(torch.stack(log_probs) * advantages).mean()

        optimizer.zero_grad(); loss.backward(); optimizer.step()

        if not board.is_game_over():
            board.push(chess.Move(torch.argmax(probs).item() // 64, torch.argmax(probs).item() % 64))
        else: board.reset()
        if i % 50 == 0: print(f"RL Iter {i} | Loss: {loss.item():.4f} | Avg Reward: {rewards.mean().item():.2f}")

In [ ]:
model = ChessPolicyNet().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
engine = chess.engine.SimpleEngine.popen_uci(STOCKFISH_PATH)
engine.configure({"Threads": 2})

try:
    supervised_pretrain(model, optimizer, engine, iterations=3000)
    train_grpo_enhanced(model, optimizer, engine, iterations=1500)
    torch.save(model.state_dict(), "grpo_chess_sampling.pth")
finally:
    engine.quit()

--- Starting Supervised Pre-training ---
SL Iter 0000 | Loss: 8.3179
SL Iter 0100 | Loss: 8.3361
SL Iter 0200 | Loss: 9.2981
SL Iter 0300 | Loss: 8.7778
SL Iter 0400 | Loss: 8.8517
SL Iter 0500 | Loss: 9.8525
SL Iter 0600 | Loss: 3.5760
SL Iter 0700 | Loss: 9.7245
SL Iter 0800 | Loss: 1.1331
SL Iter 0900 | Loss: 2.7753
SL Iter 1000 | Loss: 1.3123
SL Iter 1100 | Loss: 4.3355
SL Iter 1200 | Loss: 5.6475
SL Iter 1300 | Loss: 6.6129
SL Iter 1400 | Loss: 9.6061
SL Iter 1500 | Loss: 1.9939
SL Iter 1600 | Loss: 1.4877
SL Iter 1700 | Loss: 4.3482
SL Iter 1800 | Loss: 4.6853
SL Iter 1900 | Loss: 4.3864
SL Iter 2000 | Loss: 5.8670
SL Iter 2100 | Loss: 4.3938
SL Iter 2200 | Loss: 4.2176
SL Iter 2300 | Loss: 3.3744
SL Iter 2400 | Loss: 3.1241
SL Iter 2500 | Loss: 3.4930
SL Iter 2600 | Loss: 4.2275
SL Iter 2700 | Loss: 8.7222
SL Iter 2800 | Loss: 4.5841
SL Iter 2900 | Loss: 6.1364

--- Starting GRPO Reinforcement Learning ---
RL Iter 0 | Loss: 0.1346 | Avg Reward: 0.31
RL Iter 50 | Loss: -0.5019 | 

In [ ]:
import chess.svg

def play_final(model):
    board = chess.Board()
    model.eval()

    # UI Elements
    move_input = widgets.Text(description='Type Move:', placeholder='e.g. e2e4')
    temp_slider = widgets.FloatSlider(value=0.7, min=0.1, max=2.0, step=0.1, description='Temp:')
    top_p_slider = widgets.FloatSlider(value=0.9, min=0.1, max=1.0, step=0.05, description='Top-p:')

    def handle_move(sender):
        move_uci = move_input.value.strip().lower()
        move_input.value = ''
        try:
            move = chess.Move.from_uci(move_uci)
            if move in board.legal_moves:
                board.push(move)
                if not board.is_game_over():
                    # AI Turn with Sampling
                    with torch.no_grad():
                        logits = model(board_to_tensor(board))
                        mask = get_legal_mask(board)
                        ai_idx = sample_move(logits, mask, temp_slider.value, top_p_slider.value)
                        board.push(chess.Move(ai_idx // 64, ai_idx % 64))
                render()
            else: print("Illegal move!")
        except: print("Invalid UCI format!")

    def render():
        clear_output(wait=True)
        display(SVG(chess.svg.board(board=board, size=400)))
        if board.is_game_over(): print(f"Result: {board.result()}")
        else:
            display(widgets.HBox([temp_slider, top_p_slider]))
            display(move_input)

    move_input.on_submit(handle_move)
    render()

play_final(model)